In [ ]:
!pip install -q ultralytics pyyaml mlflow opencv-python-headless matplotlib seaborn pandas

In [ ]:
import os
import yaml
import zipfile
import shutil
import time
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import mlflow
import mlflow.pytorch
import torch
from ultralytics import YOLO

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

# ─── CONFIG CENTRALE — modifier ici uniquement ────────────────────────────────
CONFIG = {
    # Paths
    'dataset_root':    Path('/kaggle/input/datasets/mohamedsayari77/cow-behavior'),
    'working_dir':     Path('/kaggle/working'),
    'fixed_yaml':      Path('/kaggle/working/data_fixed.yaml'),
    'mlflow_uri':      'sqlite:////kaggle/working/mlruns.db',
    'experiment_name': 'Cow_Behavior_YOLOv11',

    # Benchmark (rapide)
    'benchmark_models':  ['yolo11n.pt', 'yolo11s.pt', 'yolo11m.pt'],
    'benchmark_epochs':  30,
    'benchmark_patience': 10,

    # Entraînement final
    'final_epochs':    120,
    'final_patience':  25,
    'imgsz':           768,
    'batch':           8,
    'optimizer':       'AdamW',
    'lr0':             0.001,
    'weight_decay':    0.0005,

    # Augmentations
    'hsv_h':      0.015,
    'hsv_s':      0.7,
    'hsv_v':      0.4,
    'degrees':    10.0,
    'translate':  0.1,
    'scale':      0.5,
    'flipud':     0.1,
    'fliplr':     0.5,
    'mosaic':     1.0,
    'mixup':      0.1,
    'copy_paste': 0.1,
    'close_mosaic': 10,

    # Inférence
    'conf_threshold': 0.25,
    'iou_threshold':  0.45,
}

print('✅ Configuration chargée')
print(f"   Dataset : {CONFIG['dataset_root']}")
print(f"   Working : {CONFIG['working_dir']}")
CONFIG['working_dir'].mkdir(parents=True, exist_ok=True)
# Auto-detect GPU/CPU
import torch as _torch
DEVICE = 0 if _torch.cuda.is_available() else 'cpu'
_HAS_GPU = _torch.cuda.is_available()
if not _HAS_GPU:
    CONFIG['benchmark_epochs']   = 5
    CONFIG['benchmark_patience'] = 3
    CONFIG['final_epochs']       = 10
    CONFIG['final_patience']     = 5
    CONFIG['imgsz']              = 640
    CONFIG['batch']              = 4
    print('WARNING: GPU absent - epochs reduits')
else:
    print('GPU OK - training sur cuda:0')


In [ ]:
def audit_split(root: Path, split: str) -> dict:
    """Audit un split : compte images, labels, labels vides."""
    images_dir = root / split / 'images'
    labels_dir = root / split / 'labels'

    image_files = sorted(images_dir.glob('*.*')) if images_dir.exists() else []
    label_files  = sorted(labels_dir.glob('*.txt')) if labels_dir.exists() else []

    empty_labels   = [f for f in label_files if f.stat().st_size == 0]
    missing_labels = []
    for img in image_files:
        lbl = labels_dir / (img.stem + '.txt')
        if not lbl.exists():
            missing_labels.append(img.name)

    return {
        'split': split,
        'images': len(image_files),
        'labels': len(label_files),
        'empty_labels': len(empty_labels),
        'missing_labels': len(missing_labels),
        'image_files': image_files,
        'label_files': label_files,
    }


def load_class_names(yaml_path: Path) -> list:
    with open(yaml_path, 'r', encoding='utf-8') as f:
        d = yaml.safe_load(f)
    return d.get('names', [])


def count_class_distribution(splits_data: list, class_names: list) -> pd.DataFrame:
    """Compte les occurrences de chaque classe dans tous les labels."""
    counts = {name: 0 for name in class_names}
    for split_info in splits_data:
        for lbl_file in split_info['label_files']:
            try:
                lines = lbl_file.read_text().strip().splitlines()
                for line in lines:
                    if line.strip():
                        cls_id = int(line.split()[0])
                        if cls_id < len(class_names):
                            counts[class_names[cls_id]] += 1
            except Exception:
                pass
    return pd.DataFrame(list(counts.items()), columns=['class', 'count']).sort_values('count', ascending=False)


root = CONFIG['dataset_root']

if not root.exists():
    raise FileNotFoundError(f'Dataset introuvable : {root}')

# Auto-detect data.yaml (handles subfolders like "Nouveau dossier/")
yaml_candidates = list(root.rglob('data.yaml'))
if not yaml_candidates:
    raise FileNotFoundError(f'data.yaml introuvable dans : {root}')
yaml_path = yaml_candidates[0]
root = yaml_path.parent   # real dataset root (where train/valid/test folders are)
CONFIG['dataset_root'] = root
print(f'Dataset root detecte : {root}')
print(f'data.yaml : {yaml_path}')

class_names = load_class_names(yaml_path)
print(f'Classes ({len(class_names)}) : {class_names}\n')

splits_data = [audit_split(root, s) for s in ['train', 'valid', 'test']]

# Tableau récapitulatif
summary = pd.DataFrame([{
    'Split':          s['split'],
    'Images':         s['images'],
    'Labels':         s['labels'],
    'Labels vides':   s['empty_labels'],
    'Labels manquants': s['missing_labels'],
} for s in splits_data])
print(summary.to_string(index=False))

In [ ]:
# ── Distribution des classes ───────────────────────────────────────────────────
class_df = count_class_distribution(splits_data, class_names)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
ax = axes[0]
colors = plt.cm.tab20(np.linspace(0, 1, len(class_df)))
bars = ax.barh(class_df['class'], class_df['count'], color=colors)
ax.set_xlabel('Nombre d\'annotations', fontsize=12)
ax.set_title('Distribution des classes (toutes splits)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, class_df['count']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
ax.invert_yaxis()

# Imbalance ratio
ax2 = axes[1]
max_count = class_df['count'].max()
class_df['imbalance_ratio'] = max_count / class_df['count'].replace(0, 1)
colors2 = ['#e74c3c' if r > 3 else '#2ecc71' for r in class_df['imbalance_ratio']]
ax2.bar(range(len(class_df)), class_df['imbalance_ratio'], color=colors2)
ax2.set_xticks(range(len(class_df)))
ax2.set_xticklabels(class_df['class'], rotation=45, ha='right', fontsize=8)
ax2.axhline(y=3, color='orange', linestyle='--', label='Seuil imbalance (3x)')
ax2.set_title('Ratio déséquilibre par classe', fontsize=13, fontweight='bold')
ax2.set_ylabel('Ratio (max_class / cette_classe)')
ax2.legend()

plt.tight_layout()
plt.savefig(CONFIG['working_dir'] / 'eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalanced = class_df[class_df['imbalance_ratio'] > 3]
if not imbalanced.empty:
    print(f'⚠️  Classes déséquilibrées (ratio > 3x) : {imbalanced["class"].tolist()}')
else:
    print('✅ Dataset équilibré — pas de classe fortement sous-représentée')

In [ ]:
# ── Visualisation bounding boxes sur images aléatoires ────────────────────────
def draw_bboxes(image_path: Path, label_path: Path, class_names: list) -> np.ndarray:
    """Dessine les bounding boxes YOLO sur une image."""
    img = cv2.imread(str(image_path))
    if img is None:
        return np.zeros((300, 300, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    colors_map = plt.cm.tab20(np.linspace(0, 1, len(class_names)))

    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            color = tuple((np.array(colors_map[cls_id % len(class_names)][:3]) * 255).astype(int).tolist())
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            cv2.putText(img, label, (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return img


train_info = splits_data[0]
sample_imgs = random.sample(train_info['image_files'], min(9, len(train_info['image_files'])))

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
fig.suptitle('Échantillon images + bounding boxes (Train)', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), sample_imgs):
    lbl_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    annotated = draw_bboxes(img_path, lbl_path, class_names)
    ax.imshow(annotated)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis('off')

for ax in axes.flatten()[len(sample_imgs):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(CONFIG['working_dir'] / 'eda_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Distribution des tailles d'images ─────────────────────────────────────────
def get_image_sizes(image_files: list, max_samples: int = 200) -> pd.DataFrame:
    sizes = []
    for p in image_files[:max_samples]:
        img = cv2.imread(str(p))
        if img is not None:
            sizes.append({'width': img.shape[1], 'height': img.shape[0], 'file': p.name})
    return pd.DataFrame(sizes)

all_images = train_info['image_files'] + splits_data[1]['image_files']
sizes_df = get_image_sizes(all_images, max_samples=300)

if not sizes_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(sizes_df['width'], bins=20, color='steelblue', alpha=0.7, label='Width')
    axes[0].hist(sizes_df['height'], bins=20, color='salmon', alpha=0.7, label='Height')
    axes[0].set_title('Distribution tailles images (px)', fontweight='bold')
    axes[0].set_xlabel('Pixels')
    axes[0].legend()

    axes[1].scatter(sizes_df['width'], sizes_df['height'], alpha=0.4, s=20, color='purple')
    axes[1].set_title('Largeur vs Hauteur', fontweight='bold')
    axes[1].set_xlabel('Largeur (px)')
    axes[1].set_ylabel('Hauteur (px)')

    plt.tight_layout()
    plt.savefig(CONFIG['working_dir'] / 'eda_image_sizes.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f"Taille min  : {int(sizes_df['width'].min())} x {int(sizes_df['height'].min())}")
    print(f"Taille max  : {int(sizes_df['width'].max())} x {int(sizes_df['height'].max())}")
    print(f"Taille moy  : {int(sizes_df['width'].mean())} x {int(sizes_df['height'].mean())}")

In [ ]:
def fix_yaml(original_yaml_path: Path, output_path: Path, dataset_root: Path) -> dict:
    """Corrige les chemins du YAML et valide l'existence des dossiers."""
    with open(original_yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

    splits = {
        'train': dataset_root / 'train' / 'images',
        'val':   dataset_root / 'valid' / 'images',
        'test':  dataset_root / 'test'  / 'images',
    }

    for key, path in splits.items():
        if not path.exists():
            print(f'  ⚠️  {key} path inexistant : {path}')
        data[key] = str(path)

    with open(output_path, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, allow_unicode=True)

    print(f'✅ YAML corrigé sauvegardé : {output_path}')
    return data


fixed_yaml_data = fix_yaml(
    original_yaml_path=CONFIG['dataset_root'] / 'data.yaml',
    output_path=CONFIG['fixed_yaml'],
    dataset_root=CONFIG['dataset_root'],
)

print('\nYAML final :')
for k, v in fixed_yaml_data.items():
    print(f'  {k}: {v}')

In [ ]:
mlflow.set_tracking_uri(CONFIG['mlflow_uri'])
mlflow.set_experiment(CONFIG['experiment_name'])


def benchmark_model(model_name: str, yaml_path: str, epochs: int, patience: int) -> dict:
    """Entraîne un modèle en benchmark rapide et retourne ses métriques."""
    print(f'\n🔍 Benchmark {model_name}...')
    run_name = f'benchmark_{model_name.replace(".pt", "")}'.replace('-', '_')
    project_dir = str(CONFIG['working_dir'])

    t0 = time.time()
    model = YOLO(model_name)

    with mlflow.start_run(run_name=run_name):
        results = model.train(
            data=yaml_path,
            epochs=epochs,
            patience=patience,
            imgsz=CONFIG['imgsz'],
            batch=CONFIG['batch'],
            optimizer=CONFIG['optimizer'],
            lr0=CONFIG['lr0'],
            device=DEVICE,
            amp=_HAS_GPU,
            cache=_HAS_GPU,
            close_mosaic=5,
            workers=2,
            project=project_dir,
            name=run_name,
            exist_ok=True,
            verbose=False,
        )
        elapsed = time.time() - t0

        # Métriques finales
        map50    = float(results.results_dict.get('metrics/mAP50(B)', 0))
        map5095  = float(results.results_dict.get('metrics/mAP50-95(B)', 0))
        n_params = sum(p.numel() for p in model.model.parameters()) / 1e6

        mlflow.log_params({
            'model':    model_name,
            'epochs':   epochs,
            'imgsz':    CONFIG['imgsz'],
            'batch':    CONFIG['batch'],
            'optimizer': CONFIG['optimizer'],
        })
        mlflow.log_metrics({
            'mAP50':         map50,
            'mAP50_95':      map5095,
            'train_time_s':  round(elapsed, 1),
            'params_M':      round(n_params, 2),
        })

    best_pt = Path(project_dir) / run_name / 'weights' / 'best.pt'
    return {
        'model':        model_name,
        'mAP50':        round(map50, 4),
        'mAP50-95':     round(map5095, 4),
        'train_time_s': round(elapsed, 1),
        'params_M':     round(n_params, 2),
        'best_pt':      best_pt,
    }


benchmark_results = []
for model_name in CONFIG['benchmark_models']:
    try:
        res = benchmark_model(
            model_name,
            str(CONFIG['fixed_yaml']),
            CONFIG['benchmark_epochs'],
            CONFIG['benchmark_patience'],
        )
        benchmark_results.append(res)
    except Exception as e:
        print(f'  ❌ {model_name} échoué : {e}')

bench_df = pd.DataFrame(benchmark_results) if benchmark_results else pd.DataFrame()
print('\n' + '='*65)
print('  BENCHMARK RESULTS')
print('='*65)
if not bench_df.empty:
    print(bench_df[['model', 'mAP50', 'mAP50-95', 'train_time_s', 'params_M']].to_string(index=False))
else:
    print('Benchmark vide - active GPU (Settings > Accelerator > GPU T4 x2)')
    best_model = CONFIG['benchmark_models'][0]

In [ ]:
# ── Visualisation benchmark ────────────────────────────────────────────────────
if not bench_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Benchmark Multi-Modèles YOLOv11', fontsize=14, fontweight='bold')

    model_labels = [m.replace('.pt', '') for m in bench_df['model']]
    colors = ['#3498db', '#2ecc71', '#e74c3c']

    axes[0].bar(model_labels, bench_df['mAP50'], color=colors[:len(bench_df)])
    axes[0].set_title('mAP50')
    axes[0].set_ylim(0, 1)
    for i, v in enumerate(bench_df['mAP50']):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

    axes[1].bar(model_labels, bench_df['mAP50-95'], color=colors[:len(bench_df)])
    axes[1].set_title('mAP50-95')
    axes[1].set_ylim(0, 1)
    for i, v in enumerate(bench_df['mAP50-95']):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

    axes[2].scatter(bench_df['params_M'], bench_df['mAP50'], s=200, c=colors[:len(bench_df)], zorder=5)
    for i, row in bench_df.iterrows():
        axes[2].annotate(row['model'].replace('.pt', ''),
                         (row['params_M'], row['mAP50']),
                         textcoords='offset points', xytext=(5, 5))
    axes[2].set_xlabel('Paramètres (M)')
    axes[2].set_ylabel('mAP50')
    axes[2].set_title('Précision vs Taille modèle')

    plt.tight_layout()
    plt.savefig(CONFIG['working_dir'] / 'benchmark_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

# Choisir le meilleur modèle
best_row    = bench_df.loc[bench_df['mAP50'].idxmax()]
best_model  = best_row['model']
print(f'\n🏆 Meilleur modèle pour entraînement final : {best_model}')
print(f'   mAP50 = {best_row["mAP50"]:.4f} | mAP50-95 = {best_row["mAP50-95"]:.4f}')

In [ ]:
mlflow.end_run()  # close any leftover run from benchmark
print(f'🚀 Entraînement final avec {best_model} — {CONFIG["final_epochs"]} epochs')

final_model = YOLO(best_model)

with mlflow.start_run(run_name=f'FINAL_{best_model.replace(".pt", "")}'):
    mlflow.set_tag('stage', 'final_training')
    mlflow.set_tag('model', best_model)

    # Log config
    mlflow.log_params({
        'model':         best_model,
        'epochs':        CONFIG['final_epochs'],
        'imgsz':         CONFIG['imgsz'],
        'batch':         CONFIG['batch'],
        'optimizer':     CONFIG['optimizer'],
        'lr0':           CONFIG['lr0'],
        'weight_decay':  CONFIG['weight_decay'],
        'aug_mosaic':    CONFIG['mosaic'],
        'aug_mixup':     CONFIG['mixup'],
        'aug_copy_paste': CONFIG['copy_paste'],
        'aug_degrees':   CONFIG['degrees'],
    })

    final_results = final_model.train(
        data=str(CONFIG['fixed_yaml']),
        epochs=CONFIG['final_epochs'],
        patience=CONFIG['final_patience'],
        imgsz=CONFIG['imgsz'],
        batch=CONFIG['batch'],
        optimizer=CONFIG['optimizer'],
        lr0=CONFIG['lr0'],
        weight_decay=CONFIG['weight_decay'],
        device=DEVICE,
        amp=_HAS_GPU,
        cache=_HAS_GPU,
        close_mosaic=CONFIG['close_mosaic'],
        # Augmentations
        hsv_h=CONFIG['hsv_h'],
        hsv_s=CONFIG['hsv_s'],
        hsv_v=CONFIG['hsv_v'],
        degrees=CONFIG['degrees'],
        translate=CONFIG['translate'],
        scale=CONFIG['scale'],
        flipud=CONFIG['flipud'],
        fliplr=CONFIG['fliplr'],
        mosaic=CONFIG['mosaic'],
        mixup=CONFIG['mixup'],
        copy_paste=CONFIG['copy_paste'],
        workers=2,
        project=str(CONFIG['working_dir']),
        name='cow_behavior',
        exist_ok=True,
    )

    # Log métriques finales
    final_map50   = float(final_results.results_dict.get('metrics/mAP50(B)', 0))
    final_map5095 = float(final_results.results_dict.get('metrics/mAP50-95(B)', 0))
    mlflow.log_metrics({
        'final_mAP50':    final_map50,
        'final_mAP50_95': final_map5095,
    })

    # Log artifact best.pt
    best_pt_path = CONFIG['working_dir'] / 'cow_behavior' / 'weights' / 'best.pt'
    if best_pt_path.exists():
        mlflow.log_artifact(str(best_pt_path), artifact_path='weights')

print(f'\n✅ Entraînement final terminé')
print(f'   mAP50    = {final_map50:.4f}')
print(f'   mAP50-95 = {final_map5095:.4f}')

In [ ]:
run_dir = CONFIG['working_dir'] / 'cow_behavior'

# ── Courbes loss / mAP depuis results.csv ─────────────────────────────────────
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Courbes d\'entraînement — Cow Behavior YOLOv11', fontsize=14, fontweight='bold')

    metrics_to_plot = [
        ('train/box_loss',  'Box Loss (Train)',  '#e74c3c'),
        ('train/cls_loss',  'Cls Loss (Train)',  '#e67e22'),
        ('val/box_loss',    'Box Loss (Val)',     '#3498db'),
        ('val/cls_loss',    'Cls Loss (Val)',     '#2980b9'),
        ('metrics/mAP50(B)',    'mAP50',         '#2ecc71'),
        ('metrics/mAP50-95(B)', 'mAP50-95',      '#27ae60'),
    ]

    for ax, (col, title, color) in zip(axes.flatten(), metrics_to_plot):
        if col in df_res.columns:
            ax.plot(df_res['epoch'], df_res[col], color=color, linewidth=2)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
            # Marquer le meilleur
            if 'mAP' in col:
                best_epoch = df_res[col].idxmax()
                ax.axvline(x=df_res.loc[best_epoch, 'epoch'], color='red', linestyle='--', alpha=0.5)
                ax.scatter(df_res.loc[best_epoch, 'epoch'], df_res.loc[best_epoch, col],
                           color='red', s=80, zorder=5, label=f'Best: {df_res.loc[best_epoch, col]:.4f}')
                ax.legend()
        else:
            ax.set_title(f'{title} (N/A)', fontsize=10)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(CONFIG['working_dir'] / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  results.csv introuvable')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
conf_matrix_path = run_dir / 'confusion_matrix_normalized.png'
if conf_matrix_path.exists():
    img = plt.imread(str(conf_matrix_path))
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.title('Confusion Matrix Normalisée', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  confusion_matrix_normalized.png non trouvée — disponible après validation')

In [ ]:
# ── F1-Confidence Curve + seuil optimal ───────────────────────────────────────
f1_curve_path = run_dir / 'F1_curve.png'
pr_curve_path = run_dir / 'PR_curve.png'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, path, title in [
    (axes[0], f1_curve_path, 'F1-Confidence Curve'),
    (axes[1], pr_curve_path, 'Precision-Recall Curve'),
]:
    if path.exists():
        img = plt.imread(str(path))
        ax.imshow(img)
        ax.set_title(title, fontweight='bold')
        ax.axis('off')
    else:
        ax.set_title(f'{title} (N/A)')
        ax.text(0.5, 0.5, 'Fichier non trouvé', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(CONFIG['working_dir'] / 'f1_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Lire le seuil optimal depuis les résultats
try:
    loaded_model = YOLO(str(CONFIG['working_dir'] / 'cow_behavior' / 'weights' / 'best.pt'))
    val_res = loaded_model.val(data=str(CONFIG['fixed_yaml']), split='val', verbose=False)
    opt_conf = float(val_res.results_dict.get('fitness', CONFIG['conf_threshold']))
    print(f'✅ Seuil de confiance recommandé : {CONFIG["conf_threshold"]}')
except Exception as e:
    print(f'  Info seuil non disponible : {e}')

In [ ]:
# ── Per-class mAP50 ───────────────────────────────────────────────────────────
try:
    val_metrics = loaded_model.val(data=str(CONFIG['fixed_yaml']), split='val', verbose=False)
    class_map50 = val_metrics.box.maps  # mAP50-95 per class

    per_class_df = pd.DataFrame({
        'class': class_names[:len(class_map50)],
        'mAP50-95': [round(float(v), 4) for v in class_map50],
    }).sort_values('mAP50-95', ascending=False)

    print('\nPer-class mAP50-95 :')
    print(per_class_df.to_string(index=False))

    # Visualisation
    plt.figure(figsize=(10, 5))
    colors = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.75 else '#2ecc71'
              for v in per_class_df['mAP50-95']]
    bars = plt.barh(per_class_df['class'], per_class_df['mAP50-95'], color=colors)
    plt.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Seuil 0.5')
    plt.axvline(x=0.75, color='orange', linestyle='--', alpha=0.5, label='Seuil 0.75')
    plt.xlabel('mAP50-95')
    plt.title('Performance par classe', fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(CONFIG['working_dir'] / 'per_class_map.png', dpi=150, bbox_inches='tight')
    plt.show()

    weak = per_class_df[per_class_df['mAP50-95'] < 0.5]
    if not weak.empty:
        print(f'\n⚠️  Classes faibles (mAP < 0.5) : {weak["class"].tolist()}')
except Exception as e:
    print(f'Per-class metrics non disponibles : {e}')

In [ ]:
print('🔬 Validation finale sur le TEST SET (jamais vu pendant l\'entraînement)\n')

try:
    test_metrics = loaded_model.val(
        data=str(CONFIG['fixed_yaml']),
        split='test',
        conf=CONFIG['conf_threshold'],
        iou=CONFIG['iou_threshold'],
        verbose=True,
    )

    test_map50   = float(test_metrics.results_dict.get('metrics/mAP50(B)', 0))
    test_map5095 = float(test_metrics.results_dict.get('metrics/mAP50-95(B)', 0))

    print(f'\n📊 Résultats Test Set :')
    print(f'   mAP50    = {test_map50:.4f}  ({test_map50*100:.1f}%)')
    print(f'   mAP50-95 = {test_map5095:.4f}  ({test_map5095*100:.1f}%)')

    mlflow.end_run()
    with mlflow.start_run(run_name='TEST_SET_EVAL'):
        mlflow.log_metrics({
            'test_mAP50':    test_map50,
            'test_mAP50_95': test_map5095,
        })
except Exception as e:
    print(f'❌ Validation test échouée : {e}')

In [ ]:
test_images_dir = CONFIG['dataset_root'] / 'test' / 'images'
test_images = sorted(list(test_images_dir.glob('*.*')))[:12]

if test_images:
    pred_results = loaded_model.predict(
        source=[str(p) for p in test_images],
        conf=CONFIG['conf_threshold'],
        iou=CONFIG['iou_threshold'],
        save=False,
        verbose=False,
    )

    n_cols = 4
    n_rows = (len(pred_results) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
    fig.suptitle('Prédictions sur Test Set', fontsize=14, fontweight='bold')
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, (result, img_path) in zip(axes_flat, zip(pred_results, test_images)):
        # Dessiner les prédictions
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        colors_map = plt.cm.tab20(np.linspace(0, 1, len(class_names)))

        h, w = img.shape[:2]
        boxes = result.boxes
        n_det = len(boxes)

        if boxes is not None and n_det > 0:
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                conf   = float(box.conf[0])
                cls_id = int(box.cls[0])
                color  = tuple((np.array(colors_map[cls_id % len(class_names)][:3]) * 255).astype(int).tolist())
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = f"{class_names[cls_id] if cls_id < len(class_names) else cls_id} {conf:.2f}"
                cv2.putText(img, label, (x1, max(y1-5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

        ax.imshow(img)
        ax.set_title(f"{img_path.name[:25]}\n{n_det} détection(s)", fontsize=8)
        ax.axis('off')

    for ax in axes_flat[len(pred_results):]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(CONFIG['working_dir'] / 'test_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Pas d\'images test disponibles')

In [ ]:
best_pt = CONFIG['working_dir'] / 'cow_behavior' / 'weights' / 'best.pt'
export_model = YOLO(str(best_pt))

# ── Export ONNX ───────────────────────────────────────────────────────────────
print('📦 Export ONNX (pour déploiement FastAPI)...')
try:
    onnx_path = export_model.export(
        format='onnx',
        imgsz=CONFIG['imgsz'],
        dynamic=True,
        simplify=True,
    )
    print(f'  ✅ ONNX exporté : {onnx_path}')
except Exception as e:
    print(f'  ❌ ONNX export échoué : {e}')
    onnx_path = None

# ── Export TorchScript ────────────────────────────────────────────────────────
print('\n📦 Export TorchScript (backup)...')
try:
    ts_path = export_model.export(
        format='torchscript',
        imgsz=CONFIG['imgsz'],
    )
    print(f'  ✅ TorchScript exporté : {ts_path}')
except Exception as e:
    print(f'  ❌ TorchScript export échoué : {e}')
    ts_path = None

In [ ]:
# ── Vérification ONNX — même résultat que PyTorch ────────────────────────────
if onnx_path and Path(str(onnx_path)).exists():
    print('🔍 Vérification cohérence PyTorch ↔ ONNX...')
    try:
        import onnxruntime as ort

        test_img = str(test_images[0]) if test_images else None
        if test_img:
            pt_result   = loaded_model.predict(test_img, conf=0.25, verbose=False)[0]
            onnx_model  = YOLO(str(onnx_path))
            onnx_result = onnx_model.predict(test_img, conf=0.25, verbose=False)[0]

            pt_n   = len(pt_result.boxes) if pt_result.boxes else 0
            onnx_n = len(onnx_result.boxes) if onnx_result.boxes else 0
            print(f'  PyTorch détections : {pt_n}')
            print(f'  ONNX    détections : {onnx_n}')

            if abs(pt_n - onnx_n) <= 1:
                print('  ✅ Cohérence validée')
            else:
                print('  ⚠️  Léger écart — vérifier le seuil conf')
    except ImportError:
        print('  onnxruntime non installé — skip vérification')
    except Exception as e:
        print(f'  Vérification ignorée : {e}')

In [ ]:
def create_final_package(run_dir: Path, working_dir: Path, onnx_path=None) -> Path:
    """Copie tous les artefacts dans final_export/ et crée un ZIP."""
    export_dir = working_dir / 'final_export'
    export_dir.mkdir(parents=True, exist_ok=True)

    weights_dir = run_dir / 'weights'
    artifacts = {
        weights_dir / 'best.pt':      export_dir / 'best.pt',
        weights_dir / 'last.pt':      export_dir / 'last.pt',
        CONFIG['fixed_yaml']:         export_dir / 'data.yaml',
        run_dir / 'results.csv':      export_dir / 'results.csv',
        run_dir / 'results.png':      export_dir / 'results.png',
        run_dir / 'args.yaml':        export_dir / 'args.yaml',
        working_dir / 'training_curves.png':    export_dir / 'training_curves.png',
        working_dir / 'per_class_map.png':      export_dir / 'per_class_map.png',
        working_dir / 'benchmark_comparison.png': export_dir / 'benchmark_comparison.png',
        working_dir / 'test_predictions.png':   export_dir / 'test_predictions.png',
        working_dir / 'eda_class_distribution.png': export_dir / 'eda_class_distribution.png',
    }

    # ONNX si disponible
    if onnx_path and Path(str(onnx_path)).exists():
        artifacts[Path(str(onnx_path))] = export_dir / 'best.onnx'

    copied = 0
    for src, dst in artifacts.items():
        if src.exists():
            shutil.copy(src, dst)
            copied += 1
        else:
            print(f'  ⚠️  Absent : {src.name}')

    print(f'\n📁 {copied}/{len(artifacts)} fichiers copiés dans {export_dir}')

    # ZIP
    zip_path = working_dir / 'Model_cow_behavior_YOLOv11.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in export_dir.rglob('*'):
            if f.is_file():
                zf.write(f, arcname=f.relative_to(export_dir))

    size_mb = zip_path.stat().st_size / 1024**2
    print(f'✅ ZIP prêt : {zip_path} ({size_mb:.1f} MB)')
    return zip_path


zip_file = create_final_package(
    run_dir=CONFIG['working_dir'] / 'cow_behavior',
    working_dir=CONFIG['working_dir'],
    onnx_path=onnx_path,
)

In [ ]:
# ── Résumé final ──────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('  RÉSUMÉ FINAL — Cow Behavior Detection (Lie / Stand / Walk)')
print('='*65)
print(f"  Modèle           : {best_model}")
print(f"  Classes          : {len(class_names)} ({', '.join(class_names[:3])}...)")
print(f"  mAP50 (val)      : {final_map50:.4f} ({final_map50*100:.1f}%)")
print(f"  mAP50-95 (val)   : {final_map5095:.4f} ({final_map5095*100:.1f}%)")
try:
    print(f"  mAP50 (test)     : {test_map50:.4f} ({test_map50*100:.1f}%)")
except NameError:
    pass
print(f"  Seuil conf       : {CONFIG['conf_threshold']}")
print(f"  Image size       : {CONFIG['imgsz']}px")
print(f"  ONNX exporté     : {'Oui' if onnx_path else 'Non'}")
print(f"  ZIP              : {zip_file.name}")
print('='*65)
print('\n🚀 Utilisation FastAPI :')
print('   from ultralytics import YOLO')
print('   model = YOLO("best.pt")  # ou best.onnx pour ONNX Runtime')
print('   results = model.predict(image, conf=0.25)')